# Airbnb Prices in European Cities - Full Pipeline

1. Connect the dataset files (merge 20 CSVs)
2. Handle missing values: Selenium scraping + cleaning
3. Save cleaned data (CSV + Excel)
4. SQL star schema (run `04_schema.sql` in SSMS first, then load from here)
5. Power BI dashboard (see `06_powerbi_guide.md`)

## 0. Config

In [11]:
from pathlib import Path
import re
import time
from datetime import date, timedelta

import numpy as np
import pandas as pd

RAW_DIR = Path("D:/Data/raw/archive")   # the 20 Kaggle CSV files
DATA_DIR = Path("D:/Data")              # all outputs go here
DATA_DIR.mkdir(parents=True, exist_ok=True)

RUN_SCRAPE = False   # True = run Selenium scraping (takes ~1 hour for 50 per city)
RUN_SQL = True      # True = load into SQL Server (after running 04_schema.sql)
MAX_PER_CITY = 50
NIGHTS = 2

MASTER_RAW = DATA_DIR / "master_raw.csv"
SCRAPED_RAW = DATA_DIR / "scraped_airbnb.csv"
MASTER_CLEAN = DATA_DIR / "master_clean"
SCRAPED_CLEAN = DATA_DIR / "scraped_clean"

## 1. Connect dataset files
City and day type exist only in the file names, so they are added as columns.

In [2]:
frames = []
for f in sorted(RAW_DIR.glob("*.csv")):
    city, day = f.stem.split("_")          # amsterdam_weekdays -> amsterdam, weekdays
    df = pd.read_csv(f)
    df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")
    df["city"] = city.title()
    df["day_type"] = "Weekday" if day == "weekdays" else "Weekend"
    frames.append(df)
    print(f"{f.name}: {len(df)} rows")

assert frames, f"No CSV files found in {RAW_DIR}"
master = pd.concat(frames, ignore_index=True)
print("Total rows:", len(master))
master.to_csv(MASTER_RAW, index=False)
master.head()

amsterdam_weekdays.csv: 1103 rows
amsterdam_weekends.csv: 977 rows
athens_weekdays.csv: 2653 rows
athens_weekends.csv: 2627 rows
barcelona_weekdays.csv: 1555 rows
barcelona_weekends.csv: 1278 rows
berlin_weekdays.csv: 1284 rows
berlin_weekends.csv: 1200 rows
budapest_weekdays.csv: 2074 rows
budapest_weekends.csv: 1948 rows
lisbon_weekdays.csv: 2857 rows
lisbon_weekends.csv: 2906 rows
london_weekdays.csv: 4614 rows
london_weekends.csv: 5379 rows
paris_weekdays.csv: 3130 rows
paris_weekends.csv: 3558 rows
rome_weekdays.csv: 4492 rows
rome_weekends.csv: 4535 rows
vienna_weekdays.csv: 1738 rows
vienna_weekends.csv: 1799 rows
Total rows: 51707


,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,...,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,...,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,Weekday
1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,...,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,Weekday
2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,...,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,Weekday
3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,...,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,Weekday
4,485.552926,Private room,False,True,2.0,True,0,0,10.0,98.0,...,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,Weekday


## 2. Web scraping with Selenium (extra features)
Airbnb is a React site, so `requests`/`BeautifulSoup` is not enough.
Entire home/apt with 2+ bedrooms, up to `MAX_PER_CITY` listings per city.
The original data is used only to get the city names. Scrape only where allowed by the site terms.

In [3]:
COUNTRY = {
    "Amsterdam": "Netherlands", "Athens": "Greece", "Barcelona": "Spain",
    "Berlin": "Germany", "Budapest": "Hungary", "Lisbon": "Portugal",
    "London": "United Kingdom", "Paris": "France", "Rome": "Italy",
    "Vienna": "Austria",
}


def stay_dates():
    today = date.today()
    monday = today + timedelta(days=(7 - today.weekday()) % 7 or 7)
    checkin = monday + timedelta(days=21)
    return checkin, checkin + timedelta(days=NIGHTS)


def get_cities():
    df = pd.read_csv(MASTER_RAW, usecols=["city", "room_type", "bedrooms"])
    df = df[(df["room_type"] == "Entire home/apt") & (df["bedrooms"] == 2)]
    return sorted(df["city"].unique())


def search_url(city):
    cin, cout = stay_dates()
    place = f"{city}--{COUNTRY.get(city, '')}".strip("-")
    return (
        f"https://www.airbnb.com/s/{place}/homes"
        f"?room_types%5B%5D=Entire%20home%2Fapt&min_bedrooms=2"
        f"&checkin={cin}&checkout={cout}&adults=2&currency=EUR"
    )


def parse_price(card_text):
    """Price PER NIGHT (EUR) from a search card text, or None."""
    t = card_text.replace("\u00a0", " ")
    m = re.findall(r"[€$£]\s?([\d,]+)\s*(?:EUR)?\s*(?:for|total)\s*(\d+)?\s*nights?", t, flags=re.I)
    if m:
        amount, n = m[-1]
        n = int(n) if n else NIGHTS
        return round(float(amount.replace(",", "")) / n, 2)
    m = re.findall(r"[€$£]\s?([\d,]+)\s*(?:per\s*)?night", t, flags=re.I)
    if m:
        return float(m[-1].replace(",", ""))
    m = re.findall(r"[€$£]\s?([\d,]+)", t)
    if m:
        return round(float(m[-1].replace(",", "")) / NIGHTS, 2)
    return None

In [4]:
def run_scraping():
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC

    opts = Options()
    opts.add_argument("--start-maximized")
    opts.add_argument("--lang=en-US")
    driver = webdriver.Chrome(options=opts)

    def collect_cards(city, limit):
        driver.get(search_url(city))
        time.sleep(7)
        found = {}
        while len(found) < limit:
            for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/rooms/']"):
                href = (a.get_attribute("href") or "").split("?")[0]
                if not href or href in found:
                    continue
                card = driver.execute_script(
                    "return arguments[0].closest('[data-testid=\"card-container\"]')"
                    " || arguments[0].closest('[itemprop=\"itemListElement\"]');", a)
                found[href] = parse_price(card.text if card is not None else "")
            nxt = driver.find_elements(By.CSS_SELECTOR, "a[aria-label='Next']")
            if len(found) >= limit or not nxt:
                break
            driver.execute_script("arguments[0].click();", nxt[0])
            time.sleep(6)
        return list(found.items())[:limit]

    def scrape_listing(url, city, price):
        driver.get(url)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))
        time.sleep(4)
        body = driver.find_element(By.TAG_NAME, "body").text
        name = driver.find_element(By.TAG_NAME, "h1").text
        rating = reviews = None
        m = re.search(r"(\d\.\d{1,2})\s*[·•]\s*([\d,]+)\s*reviews?", body)
        if m:
            rating, reviews = float(m.group(1)), int(m.group(2).replace(",", ""))
        else:
            m = re.search(r"Rated\s+(\d\.\d{1,2})\s+out of 5", body)
            if m:
                rating = float(m.group(1))
            m = re.search(r"\b([\d,]+)\s+reviews?\b", body)
            if m:
                reviews = int(m.group(1).replace(",", ""))
            if re.search(r"\bNew\b", body[:1500]) and reviews is None:
                reviews = 0
        is_super = bool(re.search(r"superhost\s*[·•]\s*\d+\s*(?:year|month)", body, flags=re.I)
                        or re.search(r"is a superhost", body, flags=re.I))
        is_fav = "guest favorite" in body[:2000].lower() or "one of the most loved homes" in body.lower()
        return {"city": city, "property_name": name, "price": price, "rating": rating,
                "num_reviews": reviews, "is_superhost": is_super,
                "is_guest_favorite": is_fav, "property_url": url}

    rows = []
    try:
        for city in get_cities():
            print("==", city)
            try:
                cards = collect_cards(city, MAX_PER_CITY)
            except Exception as e:
                print("  could not collect:", e)
                continue
            print(f"  {len(cards)} listings")
            for i, (url, price) in enumerate(cards, 1):
                try:
                    rows.append(scrape_listing(url, city, price))
                except Exception as e:
                    print(f"  skip {url}: {type(e).__name__}")
                time.sleep(2)
                if i % 10 == 0:
                    pd.DataFrame(rows).to_csv(DATA_DIR / "scraped_checkpoint.csv", index=False)
    finally:
        driver.quit()
        out = pd.DataFrame(rows)
        try:
            out.to_csv(SCRAPED_RAW, index=False)
            print("Saved", len(out), "rows to", SCRAPED_RAW)
        except PermissionError:
            backup = SCRAPED_RAW.with_name("scraped_airbnb_backup.csv")
            out.to_csv(backup, index=False)
            print("File was open, saved to", backup)


if RUN_SCRAPE:
    run_scraping()
else:
    print("RUN_SCRAPE = False -> using existing", SCRAPED_RAW)

RUN_SCRAPE = False -> using existing D:\Data\scraped_airbnb.csv


## 3. Clean the data
Original data: rename columns, nulls -> city median / mode, drop duplicates and extreme price outliers.
Scraped data: missing rating = new listing (kept NULL), missing reviews = 0, missing price = city median.
Raw files are never overwritten; cleaned data is saved as `*_clean.csv` / `*_clean.xlsx`.

In [5]:
m = pd.read_csv(MASTER_RAW)
print("Nulls (original):")
print(m.isna().sum()[m.isna().sum() > 0])

m = m.drop_duplicates()
m = m.rename(columns={
    "realSum": "price",
    "host_is_superhost": "is_superhost",
    "multi": "is_multi",
    "biz": "is_biz",
    "dist": "dist_center",
    "metro_dist": "dist_metro",
    "guest_satisfaction_overall": "guest_satisfaction",
})
for c in m.select_dtypes("number").columns:
    m[c] = m[c].fillna(m.groupby("city")[c].transform("median"))
for c in ["room_type", "day_type"]:
    m[c] = m[c].fillna(m[c].mode()[0])
for c in ["room_shared", "room_private", "is_superhost", "is_multi", "is_biz"]:
    m[c] = m[c].astype(bool)
m["price"] = m["price"].round(2)
cap = m.groupby("city")["price"].transform(lambda s: s.quantile(0.995))
m = m[m["price"] <= cap].reset_index(drop=True)
print("Original clean rows:", len(m))
m.head()

Nulls (original):
Series([], dtype: int64)
Original clean rows: 51445


,price,room_type,room_shared,room_private,person_capacity,is_superhost,is_multi,is_biz,cleanliness_rating,guest_satisfaction,...,dist_center,dist_metro,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type
0,194.03,Private room,False,True,2.0,False,True,False,10.0,93.0,...,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,Amsterdam,Weekday
1,344.25,Private room,False,True,4.0,False,False,False,8.0,85.0,...,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,Amsterdam,Weekday
2,264.10,Private room,False,True,2.0,False,False,True,9.0,87.0,...,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,Amsterdam,Weekday
3,433.53,Private room,False,True,4.0,False,False,True,9.0,90.0,...,0.384862,0.439876,493.272534,26.119108,875.033098,60.973565,4.89417,52.37663,Amsterdam,Weekday
4,485.55,Private room,False,True,2.0,True,False,False,10.0,98.0,...,0.544738,0.318693,552.830324,29.272733,815.305740,56.811677,4.90051,52.37508,Amsterdam,Weekday


In [6]:
s = pd.read_csv(SCRAPED_RAW)
print("Nulls (scraped):")
print(s.isna().sum())

s = s.drop_duplicates(subset="property_url")
s["is_superhost"] = s["is_superhost"].astype(bool)
s["is_guest_favorite"] = s["is_guest_favorite"].astype(bool)
s["num_reviews"] = s["num_reviews"].fillna(0).astype(int)
s["price"] = s["price"].fillna(s.groupby("city")["price"].transform("median"))
s["rating"] = s["rating"].where(s["rating"].between(0, 5))
s["room_type"] = "Entire home/apt"
s["bedrooms"] = 2
print("Scraped clean rows:", len(s))
s.head()

Nulls (scraped):
city                  0
property_name         0
price                 0
rating               29
num_reviews           0
is_superhost          0
is_guest_favorite     0
property_url          0
dtype: int64
Scraped clean rows: 500


,city,property_name,price,rating,num_reviews,is_superhost,is_guest_favorite,property_url,room_type,bedrooms
0,Amsterdam,Home at the Park for 4 people,267.5,4.82,349,True,False,https://www.airbnb.com/rooms/36669987,Entire home/apt,2
1,Amsterdam,Luxury African Wellness Retreat near Amsterdam,233.0,4.96,105,True,True,https://www.airbnb.com/rooms/1496485244294966674,Entire home/apt,2
2,Amsterdam,Family house in the nature near Amsterdam&Airport,171.0,5.00,4,True,False,https://www.airbnb.com/rooms/1764465033820737525,Entire home/apt,2
3,Amsterdam,DREAMCATCHER private place private gardenAmste...,143.5,4.85,471,False,False,https://www.airbnb.com/rooms/23845191,Entire home/apt,2
4,Amsterdam,Lovely private cottage near Amsterdam,265.5,4.99,158,False,True,https://www.airbnb.com/rooms/771572476281780953,Entire home/apt,2


In [7]:
m.to_csv(f"{MASTER_CLEAN}.csv", index=False)
s.to_csv(f"{SCRAPED_CLEAN}.csv", index=False)
m.to_excel(f"{MASTER_CLEAN}.xlsx", index=False)
s.to_excel(f"{SCRAPED_CLEAN}.xlsx", index=False)
print("Saved cleaned files in", DATA_DIR)

Saved cleaned files in D:\Data


## 4. Quick checks before the warehouse

In [8]:
print(m.groupby(["city", "day_type"])["price"].mean().round(1).unstack())
print()
print(s.groupby("city")[["price", "rating", "num_reviews"]].mean().round(2))

day_type   Weekday  Weekend
city                       
Amsterdam    529.5    585.3
Athens       146.1    145.3
Barcelona    274.4    275.8
Berlin       229.6    237.0
Budapest     162.4    179.2
Lisbon       233.6    236.7
London       337.0    345.2
Paris        383.0    377.1
Rome         196.6    204.3
Vienna       223.4    232.5

            price  rating  num_reviews
city                                  
Amsterdam  277.49    4.82       205.12
Athens     109.75    4.89       184.18
Barcelona  194.22    4.74       829.56
Berlin     185.35    4.84       170.14
Budapest   118.60    4.90       186.32
Lisbon     166.38    4.88       190.58
London     301.28    4.89        39.90
Paris      331.65    4.89       106.10
Rome       199.74    4.90       174.38
Vienna     199.66    4.88       125.82


In [9]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'PostgreSQL ANSI(x64)', 'PostgreSQL ANSI', 'PostgreSQL Unicode(x64)', 'PostgreSQL Unicode', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


## 5. Load into SQL Server (star schema)
Run `04_schema.sql` in SSMS first, then set `RUN_SQL = True` in the config cell and edit the connection string.
`Dim_City` is shared by both fact tables (the scraped data has no key shared with the original listings).

In [12]:
def load_to_sql():
    from sqlalchemy import create_engine, text

    CONN = ("mssql+pyodbc://@ASMAAHMED/AirbnbDW"
        "?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes")
    engine = create_engine(CONN, fast_executemany=True)

    def load_dim(df, table):
        df.drop_duplicates().reset_index(drop=True).to_sql(table, engine, if_exists="append", index=False)
        return pd.read_sql(f"SELECT * FROM {table}", engine)

    with engine.begin() as con:
        for t in ["Fact_Scraped_Listing", "Fact_Listing_Price", "Dim_Host",
                  "Dim_RoomType", "Dim_DayType", "Dim_City"]:
            con.execute(text(f"DELETE FROM {t}"))

    cities = sorted(set(m["city"]) | set(s["city"]))
    dim_city = load_dim(pd.DataFrame({"city_name": cities,
                                      "country": [COUNTRY.get(c) for c in cities]}), "Dim_City")
    dim_day = load_dim(m[["day_type"]], "Dim_DayType")
    dim_room = load_dim(pd.concat([m[["room_type", "room_shared", "room_private"]],
                                   pd.DataFrame({"room_type": ["Entire home/apt"],
                                                 "room_shared": [False], "room_private": [False]})]),
                        "Dim_RoomType")
    dim_host = load_dim(m[["is_superhost", "is_multi", "is_biz"]], "Dim_Host")
    for d, cols in [(dim_room, ["room_shared", "room_private"]),
                    (dim_host, ["is_superhost", "is_multi", "is_biz"])]:
        d[cols] = d[cols].astype(bool)

    f = (m.merge(dim_city, left_on="city", right_on="city_name")
           .merge(dim_day, on="day_type")
           .merge(dim_room, on=["room_type", "room_shared", "room_private"])
           .merge(dim_host, on=["is_superhost", "is_multi", "is_biz"]))
    cols = ["city_key", "daytype_key", "room_key", "host_key", "price", "person_capacity",
            "bedrooms", "cleanliness_rating", "guest_satisfaction", "dist_center",
            "dist_metro", "attr_index", "attr_index_norm", "rest_index",
            "rest_index_norm", "lat", "lng"]
    f[cols].to_sql("Fact_Listing_Price", engine, if_exists="append", index=False)

    entire = int(dim_room[(dim_room.room_type == "Entire home/apt")
                          & (~dim_room.room_shared) & (~dim_room.room_private)].iloc[0]["room_key"])
    sf = s.merge(dim_city, left_on="city", right_on="city_name")
    sf["room_key"] = entire
    sf["host_key"] = None
    scols = ["city_key", "room_key", "host_key", "property_name", "price", "rating",
             "num_reviews", "is_superhost", "is_guest_favorite", "property_url"]
    sf[scols].to_sql("Fact_Scraped_Listing", engine, if_exists="append", index=False)
    print("Loaded", len(f), "original rows and", len(sf), "scraped rows")


if RUN_SQL:
    load_to_sql()
else:
    print("RUN_SQL = False -> skipped")

Loaded 51445 original rows and 500 scraped rows


In [13]:
git --version
git init
git add .
git status
git commit -m "Airbnb prices pipeline: scraping, star schema, Power BI dashboard"
git branch -M main
git remote add origin https://github.com/USERNAME/airbnb-prices-european-cities.git
git push -u origin main

SyntaxError: invalid syntax (1561544951.py, line 2)